In [25]:
import pandas as pd
import numpy as np

TARGETS = ["theta"]

In [38]:
results_1l = pd.read_excel("resultados-1l.xlsx")
results_2l = pd.read_excel("resultados-2l.xlsx")
# results_3l = pd.read_excel("resultados-3l.xlsx")

results = pd.concat(
[results_1l, results_2l],
ignore_index=True
)
results = results_2l


In [39]:
results

,model,Neurons,Ld,Lp,reg,seed,R2_ZZx1_theta,R2diff_ZZx1_theta,R2_ZZx2_theta,R2diff_ZZx2_theta,...,R2_LSG_1_theta,R2diff_LSG_1_theta,R2_LSG_2_theta,R2diff_LSG_2_theta,R2_ZZx1_inv_theta,R2diff_ZZx1_inv_theta,R2_zzx2_inv2_theta,R2diff_zzx2_inv2_theta,R2_semiCirc_theta,R2diff_semiCirc_theta
0,model_arch5-1_r0.01_Ld0.5_Lp0.5_seed8743,"[5, 1]",0.5,0.5,0.01,8743,0.770566,0.680722,0.936064,0.446065,...,-4.910094,0.367122,-2.237637,0.261603,-0.018212,0.407374,-4.755363,0.248651,-6.520297,0.120518
1,model_arch5-1_r0.01_Ld0.5_Lp0.5_seed5650,"[5, 1]",0.5,0.5,0.01,5650,0.678139,0.462674,0.489916,0.285407,...,-2.206222,0.354685,-1.751085,0.224032,-2.204715,0.204595,-6.661857,0.134031,-8.358199,0.090238
2,model_arch5-1_r0.01_Ld0.5_Lp0.5_seed3975,"[5, 1]",0.5,0.5,0.01,3975,0.592197,0.476542,0.466432,0.292847,...,-2.463324,0.361346,-2.261413,0.216216,-1.998300,0.178312,-7.600679,0.107626,-7.574987,0.097450
3,model_arch5-1_r0.01_Ld0.5_Lp0.5_seed5920,"[5, 1]",0.5,0.5,0.01,5920,0.666983,0.572598,0.901970,0.360144,...,-1.358471,0.436591,-2.337641,0.264654,-1.426557,0.288734,-8.201348,0.121232,-11.235954,0.072002
4,model_arch5-1_r0.01_Ld0.5_Lp0.5_seed3206,"[5, 1]",0.5,0.5,0.01,3206,0.796305,0.565055,0.788169,0.377713,...,-1.208083,0.423052,-3.649297,0.221666,-0.628100,0.256331,-7.009545,0.124749,-13.441513,-0.007840
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2808,model_arch51-34_r0.9_Ld0.7_Lp0.3_seed5396,"[51, 34]",0.7,0.3,0.90,5396,0.852237,0.637776,0.671629,0.443743,...,0.442025,0.514363,-2.385031,0.310942,-0.108685,0.422860,-7.249708,0.182352,-20.174313,-0.065322
2809,model_arch51-34_r0.9_Ld0.7_Lp0.3_seed7948,"[51, 34]",0.7,0.3,0.90,7948,0.851133,0.679992,0.604156,0.481698,...,0.819544,0.557216,-1.798260,0.335201,-0.516214,0.460404,-8.669291,0.237674,-30.580316,-0.245002
2810,model_arch51-34_r0.9_Ld0.7_Lp0.3_seed1763,"[51, 34]",0.7,0.3,0.90,1763,0.901634,0.698414,0.948450,0.490893,...,0.826026,0.592013,-2.007197,0.354449,-1.958553,0.469289,-10.517338,0.247832,-32.541694,-0.267728
2811,model_arch51-34_r0.9_Ld0.7_Lp0.3_seed3433,"[51, 34]",0.7,0.3,0.90,3433,0.931052,0.810736,0.455556,0.473875,...,0.759322,0.603132,-3.373700,0.294347,-1.862036,0.409703,-18.381932,-0.086149,-44.013835,-0.473833


In [40]:
# 🔹 categorização dos sets (baseada nos comentários originais)

SETS_CATEGORY = {
    "ZZx1":  "Train",
    "ZZx2":     "Val",
    "ZZy1":     "Test",
    "ZZy2":     "Test",
    "LSG-1":    "Test",
    "LSG-2":    "Test",
    "ZZx1-inv": "Test",
    "ZZxReto":  "Test",
    "ZZx2-inv": "Test",
    "semiCirc": "Test",
}

def col_name(s, target):
    # sanitiza "-" pra "_" pra bater com o nome real da coluna, se for o caso
    return f"R2_{s.replace('-', '_')}_{target}"

best_models_tables = {}
N = 5  # top modelos

w_val = 0.33
w_train = 0.33
w_test = 0.33

for target in TARGETS:

    # 🔹 sets de Train, Val e Test
    train_sets = [s for s, cat in SETS_CATEGORY.items() if cat == "Train"]
    val_sets   = [s for s, cat in SETS_CATEGORY.items() if cat == "Val"]
    test_sets  = [s for s, cat in SETS_CATEGORY.items() if cat == "Test"]

    train_cols = [col_name(s, target) for s in train_sets]
    val_cols   = [col_name(s, target) for s in val_sets]
    test_cols  = [col_name(s, target) for s in test_sets]

    # 🔹 garantir que só usamos colunas existentes
    train_cols = [c for c in train_cols if c in results.columns]
    val_cols   = [c for c in val_cols if c in results.columns]
    test_cols  = [c for c in test_cols if c in results.columns]

    r2_all_cols = train_cols + val_cols + test_cols

    if not r2_all_cols:
        print(f"⚠️ Nenhuma coluna Train/Val/Test encontrada para target={target}, pulando.")
        continue

    df = results.copy()

    # 🔹 remover linhas onde QUALQUER R2 (Train/Val/Test) < 0
    # df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    df["R2_train_mean"] = df[train_cols].mean(axis=1) if train_cols else np.nan
    df["R2_val_mean"]   = df[val_cols].mean(axis=1) if val_cols else np.nan
    df["R2_test_mean"]  = df[test_cols].mean(axis=1) if test_cols else np.nan

    # =========================
    # 🔹 SCORE
    # =========================
    df["R2_std"] = df[r2_all_cols].std(axis=1)

    df["Score"] = (
        w_train * df["R2_train_mean"] +
        w_val   * df["R2_val_mean"] +
        w_test  * df["R2_test_mean"]
        - 0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    df_sorted = df.sort_values(by="Score", ascending=False)
    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    final_cols = ["model", "Neurons"] + r2_all_cols + [
        "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"
    ]
    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 5 MODELOS - theta


,model,Neurons,R2_train_mean,R2_val_mean,R2_test_mean,Score
553,model_arch70-28_r0.01_Ld0.3_Lp0.7_seed5920,"[70, 28]",0.614605,0.349674,-2.950480,-0.987902
2379,model_arch51-20_r0.9_Ld0.3_Lp0.7_seed7948,"[51, 20]",0.538894,0.541345,-3.086547,-1.035123
10,model_arch5-1_r0.01_Ld0.3_Lp0.7_seed8743,"[5, 1]",0.366632,0.513521,-3.118080,-1.102514
1717,model_arch70-48_r0.01_Ld0.3_Lp0.7_seed2614,"[70, 48]",0.665775,0.329262,-3.297696,-1.134601
109,model_arch24-8_r0.9_Ld0.3_Lp0.7_seed3206,"[24, 8]",0.580731,0.666747,-3.528385,-1.162730



📊 MÉTRICAS COMPLETAS - TOP 5 (theta)


,model,Neurons,R2_ZZx1_theta,R2_ZZx2_theta,R2_ZZy1_theta,R2_ZZy2_theta,R2_LSG_1_theta,R2_LSG_2_theta,R2_ZZx1_inv_theta,R2_ZZxReto_theta,R2_semiCirc_theta,R2_train_mean,R2_val_mean,R2_test_mean,Score
553,model_arch70-28_r0.01_Ld0.3_Lp0.7_seed5920,"[70, 28]",0.614605,0.349674,-2.947785,-9.046466,-2.155240,-1.017969,-0.017690,0.415070,-5.883279,0.614605,0.349674,-2.950480,-0.987902
2379,model_arch51-20_r0.9_Ld0.3_Lp0.7_seed7948,"[51, 20]",0.538894,0.541345,-0.818825,-11.003308,-4.955096,-1.160649,-0.495213,-0.064260,-3.108478,0.538894,0.541345,-3.086547,-1.035123
10,model_arch5-1_r0.01_Ld0.3_Lp0.7_seed8743,"[5, 1]",0.366632,0.513521,-0.661970,-9.655974,-7.467178,-0.755164,-0.843338,-0.652360,-1.790579,0.366632,0.513521,-3.118080,-1.102514
1717,model_arch70-48_r0.01_Ld0.3_Lp0.7_seed2614,"[70, 48]",0.665775,0.329262,-4.177196,-8.845032,-1.173714,-1.416584,0.140127,0.549760,-8.161230,0.665775,0.329262,-3.297696,-1.134601
109,model_arch24-8_r0.9_Ld0.3_Lp0.7_seed3206,"[24, 8]",0.580731,0.666747,-0.503078,-12.172633,-2.081949,-2.091711,-2.247960,0.110333,-5.711701,0.580731,0.666747,-3.528385,-1.162730


In [41]:
final_table.to_excel("BestModels-2l.xlsx")